<a href="https://colab.research.google.com/github/Sahilgulati2006/ai-sys-des/blob/main/04-evals/02-llm-as-judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM as Judge

**Goal:** Write judge prompts, measure agreement with human labels, and find the judge's failure modes.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client; `sentence-transformers`, `numpy` are used by this notebook.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git" sentence-transformers numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.7 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


> **🔵 Hitting a rate limit? Switch models.** This notebook is token-heavy (a generation call *and* a judge call per case, over the whole set) and Groq's free tier caps tokens-per-minute and tokens-per-day **per model**. If you get a `429`, switch to a model with a fresh budget. There are *two* knobs:
> ```python
> client, MODEL = setup(model='openai/gpt-oss-20b')   # the system-under-test (generation)
> JUDGE_MODEL = 'qwen/qwen3.6-27b'                     # set in the judge cell below
> ```
> Keep the judge a **different** model from the system under test (the self-preference lesson below). Each free model has its own limits, so pick ones with headroom from the [Groq rate-limits page](https://console.groq.com/docs/rate-limits). For a verification pass any capable chat models work; avoid the agentic `groq/compound*` models, which browse and run code.

## Why a judge at all

The deterministic checks from the last notebook stop at the surface: keywords present, right document retrieved. They can't tell you whether an answer is *faithful* (no claims beyond what the sources support) or *complete* (covers every key point of the reference). Reference answers can't be string-matched, because a correct answer can be phrased a hundred ways.

So we use a model to grade the answers. This works, but a judge is itself an LLM with its own failure modes, and most teams skip the step that makes it trustworthy: calibrating it against human judgment. We'll build the judge, then break it on purpose, then calibrate it.


## The system under test

The next two cells rebuild the RAG system from section 03 in compact form: download and clean ten IETF RFCs, chunk them by section, embed with a small local model, retrieve by cosine similarity, and answer with citations.

This is a deliberate copy, not an import. Each notebook has to run top-to-bottom in a fresh Colab runtime, and self-containment beats DRY for teaching material. If the details are unfamiliar, work through `03-rag/` first; here they're just the thing we're evaluating.


In [3]:
# Compact copy of the 03-rag pipeline. Self-containment over DRY: this notebook must run
# standalone in Colab. See 03-rag/ for the full walkthrough of every design decision here.
import os
import re
import urllib.request

RFCS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
os.makedirs('data/rfc', exist_ok=True)
for n in RFCS:
    path = f'data/rfc/rfc{n}.txt'
    if not os.path.exists(path):  # skip if cached from an earlier notebook
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)

def clean_rfc(text):
    """Strip form feeds and the page header/footer lines RFC txt files carry."""
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.search(r'\[Page \d+\]\s*$', l)          # footers
             and not re.match(r'^\s*RFC \d+.*\d{4}\s*$', l)]   # headers
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines))

def chunk_rfc(text, rfc, max_chars=2000):
    """Section-aware chunking: split on numbered headings, then cap chunk size."""
    parts = re.split(r'\n(?=\d+(?:\.\d+)*\.?\s+[A-Z])', text)
    chunks = []
    for part in parts:
        part = part.strip()
        while len(part) > max_chars:
            cut = part.rfind('\n\n', 0, max_chars)
            cut = cut if cut > 200 else max_chars
            chunks.append({'rfc': rfc, 'text': part[:cut].strip()})
            part = part[cut:].strip()
        if len(part) > 100:
            chunks.append({'rfc': rfc, 'text': part})
    return chunks

chunks = []
for n in RFCS:
    with open(f'data/rfc/rfc{n}.txt') as f:
        chunks += chunk_rfc(clean_rfc(f.read()), n)
print(f'{len(chunks)} chunks from {len(RFCS)} RFCs')

1662 chunks from 10 RFCs


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
emb = embedder.encode([c['text'] for c in chunks],
                      normalize_embeddings=True, show_progress_bar=True)

def retrieve(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = emb @ q  # cosine similarity: vectors are unit-normalized
    return [chunks[i] for i in np.argsort(-scores)[:k]]

def answer_question(question, k=5):
    """Retrieve top-k chunks, answer with citations. Returns (answer, hits)."""
    hits = retrieve(question, k)
    context = '\n\n'.join(f"[RFC {h['rfc']}]\n{h['text']}" for h in hits)
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[
            {'role': 'system',
             'content': ('Answer using ONLY the provided RFC excerpts. Cite the RFC number for '
                         'each claim, e.g. (RFC 9110). If the excerpts do not contain the answer, '
                         'say plainly that the corpus does not cover it. Do not guess.')},
            {'role': 'user', 'content': f'{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content, hits

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Same golden set as notebook 01 (self-contained copy; see that notebook for the reasoning behind each case).


In [5]:
# The golden set: ~15 cases written against the actual corpus. Each case pins down
# what a correct answer must contain (must_mention) and which RFC the retriever
# should surface (source_rfc). Unanswerable cases have source_rfc=None -- the correct
# behavior there is to decline, and hallucinating an answer is a hard failure.
GOLDEN_SET = [
    {'question': 'How does the TCP three-way handshake establish a connection?',
     'reference_answer': 'The initiator sends a SYN carrying its initial sequence number; '
                         'the peer responds with SYN-ACK carrying its own sequence number and '
                         'acknowledging the first; the initiator replies with an ACK. Both '
                         'sides now have synchronized sequence numbers.',
     'source_rfc': 793, 'must_mention': ['SYN', 'ACK', 'sequence'], 'unanswerable': False},
    {'question': 'What does the HTTP 404 status code mean?',
     'reference_answer': '404 Not Found: the origin server did not find a current '
                         'representation for the target resource, or is not willing to '
                         'disclose that one exists.',
     'source_rfc': 9110, 'must_mention': ['404', 'not found'], 'unanswerable': False},
    {'question': 'What are the four roles defined in the OAuth 2.0 framework?',
     'reference_answer': 'Resource owner, resource server, client, and authorization server.',
     'source_rfc': 6749,
     'must_mention': ['resource owner', 'client', 'authorization server', 'resource server'],
     'unanswerable': False},
    {'question': 'What is the difference between an A record and a CNAME record in DNS?',
     'reference_answer': 'An A record maps a name to a 32-bit IPv4 host address; a CNAME '
                         'record maps an alias to the canonical name of its target.',
     'source_rfc': 1035, 'must_mention': ['address', 'canonical'], 'unanswerable': False},
    {'question': 'How does a client upgrade an HTTP connection to a WebSocket connection?',
     'reference_answer': 'The client sends a GET request with Upgrade: websocket and '
                         'Connection: Upgrade headers plus a Sec-WebSocket-Key; the server '
                         'replies 101 Switching Protocols with a Sec-WebSocket-Accept value '
                         'derived from the key.',
     'source_rfc': 6455, 'must_mention': ['Upgrade', '101', 'Sec-WebSocket-Key'],
     'unanswerable': False},
    {'question': 'Which SMTP commands send a mail message, and in what order?',
     'reference_answer': 'MAIL FROM identifies the sender, one or more RCPT TO commands name '
                         'recipients, then DATA introduces the message text, terminated by a '
                         'line containing only a period.',
     'source_rfc': 5321, 'must_mention': ['MAIL', 'RCPT', 'DATA'], 'unanswerable': False},
    {'question': 'What is the purpose of the Time to Live field in the IPv4 header?',
     'reference_answer': 'TTL is an upper bound on datagram lifetime. Every module that '
                         'processes the datagram decrements it, and the datagram is discarded '
                         'when it reaches zero, so packets cannot loop forever.',
     'source_rfc': 791, 'must_mention': ['zero', 'discard'], 'unanswerable': False},
    {'question': 'What is the AS_PATH attribute in BGP and why does it matter?',
     'reference_answer': 'AS_PATH records the sequence of autonomous systems a route '
                         'advertisement has traversed. BGP uses it for loop detection (a '
                         'router rejects routes containing its own AS number) and in route '
                         'selection.',
     'source_rfc': 4271, 'must_mention': ['autonomous system', 'loop'], 'unanswerable': False},
    {'question': 'How does HTTP/2 carry multiple requests over a single connection?',
     'reference_answer': 'HTTP/2 multiplexes exchanges as independent streams over one '
                         'connection. Each frame carries a stream identifier, so concurrent '
                         'requests and responses interleave without blocking each other.',
     'source_rfc': 7540, 'must_mention': ['stream', 'frame'], 'unanswerable': False},
    {'question': 'Which HTTP methods are defined as idempotent?',
     'reference_answer': 'PUT and DELETE, plus all safe methods: GET, HEAD, OPTIONS, TRACE.',
     'source_rfc': 9110, 'must_mention': ['PUT', 'DELETE', 'GET'], 'unanswerable': False},
    {'question': 'Why does a TCP connection enter the TIME-WAIT state, and for how long?',
     'reference_answer': 'The side that closes actively waits in TIME-WAIT for twice the '
                         'maximum segment lifetime (2*MSL), so delayed segments from the old '
                         'incarnation die off before the socket pair can be reused.',
     'source_rfc': 793, 'must_mention': ['TIME-WAIT', 'MSL'], 'unanswerable': False},
    {'question': 'Walk through the OAuth 2.0 authorization code grant flow.',
     'reference_answer': 'The client redirects the resource owner to the authorization '
                         'server; after authentication and consent the server redirects back '
                         'with an authorization code; the client exchanges the code, with its '
                         'own credentials, at the token endpoint for an access token.',
     'source_rfc': 6749, 'must_mention': ['authorization code', 'access token', 'redirect'],
     'unanswerable': False},
    {'question': 'What does HTTP status 503 indicate, and which header can accompany it?',
     'reference_answer': '503 Service Unavailable: the server is currently unable to handle '
                         'the request, due to overload or maintenance. It may send Retry-After '
                         'to indicate how long to wait.',
     'source_rfc': 9110, 'must_mention': ['503', 'Retry-After'], 'unanswerable': False},
    # --- deliberately unanswerable: correct behavior is to decline ---
    {'question': 'How does QUIC combine the transport and cryptographic handshakes?',
     'reference_answer': 'Declines: QUIC (RFC 9000) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'What cipher suites does TLS 1.3 define?',
     'reference_answer': 'Declines: TLS 1.3 (RFC 8446) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'Which port does IMAP use, and how does a client select a mailbox?',
     'reference_answer': 'Declines: IMAP is not in the corpus (SMTP is the only mail '
                         'protocol here).',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
]
print(len(GOLDEN_SET), 'cases,',
      sum(c['unanswerable'] for c in GOLDEN_SET), 'unanswerable')


16 cases, 3 unanswerable


## Building the judge

A judge is an LLM, so it has LLM failure modes. The three that bite hardest, along with the design decision that counters each, line up one to one. We build the judge with all three mitigations in, then (further down) strip each one out so you can watch the failure appear:

| Judge failure mode | What it looks like | Design decision that counters it |
|---|---|---|
| **Verbosity bias** | long, padded answers outscore terse correct ones | rubric says "judge substance, not style; extra length earns nothing" |
| **Self-preference** | a model scores its *own* outputs higher | judge with a *different* model than the system under test |
| **Score compression** | everything clusters at 3–4; regressions vanish | anchor every score 1–5 to an explicit definition |

Plus two structural choices that make the grade usable at all:

- **Structured output via a forced tool call.** The judge fills a schema (`{reasoning, faithful, complete, score}`) instead of free text, so the harness aggregates without parsing prose. `tool_choice` forces the call, so we always get a grade.
- **Reasoning before score.** The schema puts `reasoning` first and the prompt says so explicitly. If the model commits to a score first, the reasoning becomes a post-hoc rationalization of a snap judgment; reasoning first measurably improves grade quality. Ordering matters.

> **⚠️ Production reality —** forcing `tool_choice` makes the model emit the call as text that the API then parses; when that parse fails, Groq returns a **400 `tool_use_failed`**, not a message. It's *intermittent* (the same input can succeed on the next try), so `judge()` wraps the call in a short retry and falls back to a sentinel score of 0 rather than crashing the whole run. That's this notebook's own lesson turned on itself: the judge is an unreliable LLM call, so build for it. A bare `.tool_calls[0]` would take down the loop on the first flaky grade.

In [6]:
import json

# Judge model: the small tier, not the big one. Two reasons. (1) Develop cheap, eval on
# target: while you iterate on the eval harness itself you will rerun it constantly, so use
# the cheap model; before a real release decision, rerun the judging pass with a
# stronger model or hand-review. (2) A different model than the system under test
# avoids self-preference bias (more on that below).
JUDGE_MODEL = 'openai/gpt-oss-20b'

JUDGE_TOOL = {
    'type': 'function',
    'function': {
        'name': 'grade_answer',
        'description': 'Record your grade for the candidate answer.',
        'parameters': {
            'type': 'object',
            'properties': {
                'reasoning': {'type': 'string',
                              'description': 'Compare candidate to reference point by point, '
                                             'BEFORE deciding any grade. 2-4 sentences.'},
                'faithful': {'type': 'boolean',
                             'description': 'True if every claim in the candidate is supported '
                                            'by the reference (nothing fabricated).'},
                'complete': {'type': 'boolean',
                             'description': 'True if the candidate covers every key point of '
                                            'the reference.'},
                'score': {'type': 'integer',
                          'description': 'Overall 1-5, anchored to the rubric.'},
            },
            'required': ['reasoning', 'faithful', 'complete', 'score'],
        },
    },
}

JUDGE_SYSTEM = """You grade answers from a retrieval-augmented QA system against a reference answer.

Rubric -- anchor your score to these definitions and use the full range:
1 = wrong or fabricated: contradicts the reference or invents facts
2 = mostly wrong: a relevant fragment, but the main point is missing or incorrect
3 = partially correct: main point present, but a real gap or a minor unsupported claim
4 = correct: matches the reference on every key point; only minor omissions
5 = correct and complete: nothing missing, nothing unsupported

If the reference says the correct behavior is to decline, a candidate that declines
scores 5 and a candidate that invents an answer scores 1.

Write your reasoning FIRST, then decide the score. Judge substance, not style:
a short correct answer outranks a long vague one. Extra length earns nothing."""

from groq import BadRequestError

def _grade_call(system, question, reference, candidate, tries=3):
    """One forced-tool judge call, retried on Groq's tool_use_failed.

    Forcing tool_choice makes the model emit the call as text that Groq then
    parses; when that parse fails it raises a 400 (tool_use_failed) instead of
    returning a message. It's intermittent, so a couple of retries clears it.
    This is the same lesson the notebook teaches, applied to the judge itself:
    an LLM call is unreliable, so wrap it. Returns the parsed dict, or None if
    every attempt failed.
    """
    for _ in range(tries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, max_tokens=700,
                tools=[JUDGE_TOOL],
                tool_choice={'type': 'function', 'function': {'name': 'grade_answer'}},
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content':
                     f'Question: {question}\n\nReference answer:\n{reference}'
                     f'\n\nCandidate answer:\n{candidate}'},
                ],
            )
            return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)
        except BadRequestError as e:
            if 'tool_use_failed' in str(e):
                continue          # the model didn't emit a valid call; try again
            raise                 # a real 400 (bad schema, etc.) -- don't hide it
    return None                   # exhausted retries; caller decides what to do

def judge(question, reference, candidate):
    """Grade one answer. Returns {reasoning, faithful, complete, score}."""
    grade = _grade_call(JUDGE_SYSTEM, question, reference, candidate)
    if grade is None:  # judge failed to produce a grade after retries -- fail safe, don't crash the run
        return {'reasoning': '(judge produced no valid tool call after retries)',
                'faithful': False, 'complete': False, 'score': 0}
    return grade

## Judging a fresh run of the RAG system

Generate answers for the full golden set, then grade each one. Note the judge gets the *reference*, not the retrieved chunks. It grades against ground truth we wrote, which is what makes the golden set load-bearing.


In [7]:
# Cost note: one generation call per case (120B, system under test) + one judge call
# per case (20B, cheap bulk judging) = 2 calls x len(GOLDEN_SET).
judged = []
for case in GOLDEN_SET:
    answer, hits = answer_question(case['question'])
    grade = judge(case['question'], case['reference_answer'], answer)
    judged.append({'question': case['question'], 'answer': answer,
                   'unanswerable': case['unanswerable'], 'grade': grade})
    print(f"score={grade['score']}  faithful={grade['faithful']}  "
          f"complete={grade['complete']}  | {case['question'][:55]}")

scores = [j['grade']['score'] for j in judged]
print('-' * 78)
print(f'mean score: {sum(scores) / len(scores):.2f}   '
      f'distribution: {sorted(scores)}')

score=5  faithful=True  complete=True  | How does the TCP three-way handshake establish a connec
score=5  faithful=True  complete=True  | What does the HTTP 404 status code mean?
score=5  faithful=True  complete=True  | What are the four roles defined in the OAuth 2.0 framew
score=1  faithful=False  complete=False  | What is the difference between an A record and a CNAME 
score=3  faithful=True  complete=False  | How does a client upgrade an HTTP connection to a WebSo
score=5  faithful=True  complete=True  | Which SMTP commands send a mail message, and in what or
score=5  faithful=True  complete=True  | What is the purpose of the Time to Live field in the IP
score=5  faithful=True  complete=True  | What is the AS_PATH attribute in BGP and why does it ma
score=5  faithful=True  complete=True  | How does HTTP/2 carry multiple requests over a single c
score=3  faithful=False  complete=False  | Which HTTP methods are defined as idempotent?
score=1  faithful=False  complete=False  | Why doe

Run this and read a few `reasoning` fields, especially on any case scored 3. The judge catches things keyword coverage can't: an answer that mentions every keyword but garbles the causality, or one that quietly adds a claim the reference doesn't support. Also look at the unanswerable cases, where the judge handles declines far more reliably than the marker-scan hack from notebook 01.

## Failure mode 1: verbosity bias

Judges prefer longer answers. Left unguarded, a padded, vague answer often outscores a terse, correct one, because length reads as effort. We constructed two answers to the 404 question: one terse and exactly right, one long, warm, and content-free (it never actually says "not found"). Judge both and compare.

In [8]:
q = 'What does the HTTP 404 status code mean?'
ref = ('404 Not Found: the origin server did not find a current representation for '
       'the target resource, or is not willing to disclose that one exists.')

terse = ('404 Not Found: the server has no current representation for the target '
         'resource, or will not disclose that one exists.')

padded = ('Great question! HTTP status codes are a fascinating and essential part of '
          'how the web communicates outcomes. The 404 code belongs to the 4xx class, '
          'which broadly indicates that something about the client\'s request was '
          'problematic in some way. Codes in this family are extremely common in '
          'day-to-day web browsing, and 404 in particular is one nearly every user '
          'has encountered. It generally relates to the situation of the requested '
          'resource on the server side of the exchange, and understanding it is an '
          'important part of debugging web applications effectively.')

for name, cand in [('terse-but-correct', terse), ('padded-but-vague', padded)]:
    g = judge(q, ref, cand)
    print(f"{name:18} score={g['score']}  faithful={g['faithful']}  "
          f"complete={g['complete']}")
    print('  ', g['reasoning'][:160])


terse-but-correct  score=5  faithful=True  complete=True
   The candidate statement reproduces the meaning of the reference verbatim: it mentions the lack of a current representation and the refusal to disclose its exist
padded-but-vague   score=3  faithful=True  complete=False
   The candidate correctly identifies that 404 is a client error (4xx class) and mentions it relates to the requested resource on the server side. However, it omit


Note the gap when you run this. With the rubric-anchored prompt ("judge substance, not style", "extra length earns nothing") the terse answer should win decisively. If you delete those two lines from `JUDGE_SYSTEM` and rerun, the padded answer's score typically climbs: the bias is real and the prompt guard is doing work. The padded answer never states the one fact that matters, yet it *sounds* like an answer, and that's exactly what un-prompted judges reward.

## Failure mode 2: self-preference

A model judging its own outputs scores them higher, since the same stylistic priors that generated the answer also grade it. That's why our judge is the 8B model while the system under test is the 70B. Note this **inverts the usual advice** that the judge should be at least as strong as the system it grades. The inversion is a deliberate trade: the 8B won't catch subtle incompleteness that the 70B would, but with a reference answer in hand, grading is much easier than answering, and a weak judge comparing candidate-to-reference still reliably catches gross failures (hallucination, missed main point, failure to decline). Develop cheap; before a release decision, rerun the judge pass on a stronger model and hand-review the disagreements.

## Failure mode 3: score compression

Judges avoid the ends of the scale. Ask for 1–5 with no definitions and you'll get a distribution that lives at 3–4, which destroys the metric's ability to detect regressions (everything is a 3.5, forever). The mitigation is already in `JUDGE_SYSTEM`: each score is *defined*. Here's the same grading with an unanchored prompt so you can see the compression directly.


In [9]:
# An unanchored judge: same schema, no rubric, no reasoning-first instruction.
# Reuses _grade_call so it gets the same tool_use_failed retry as the real judge.
def judge_unanchored(question, reference, candidate):
    grade = _grade_call('Grade the candidate answer against the reference on a 1-5 scale.',
                        question, reference, candidate)
    return grade or {'score': 0}

# Three candidates of clearly different quality for the TIME-WAIT question.
q2 = 'Why does a TCP connection enter the TIME-WAIT state, and for how long?'
ref2 = ('The side that closes actively waits in TIME-WAIT for twice the maximum '
        'segment lifetime (2*MSL), so delayed segments from the old incarnation die '
        'off before the socket pair can be reused.')
candidates = {
    'good': ('The active closer waits 2*MSL in TIME-WAIT so that stray segments '
             'from the old connection expire before the port pair is reused.'),
    'partial': ('TIME-WAIT is a state a TCP connection passes through while '
                'closing; it lasts a fixed amount of time.'),
    'wrong': ('TIME-WAIT lets the receiving side finish processing buffered data; '
              'it lasts exactly 30 seconds on all implementations.'),
}

# 6 judge calls (small tier).
print(f"{'candidate':10} {'anchored':>9} {'unanchored':>11}")
for name, cand in candidates.items():
    a = judge(q2, ref2, cand)['score']
    u = judge_unanchored(q2, ref2, cand)['score']
    print(f'{name:10} {a:>9} {u:>11}')

candidate   anchored  unanchored
good               5           5
partial            2           1
wrong              1           1


## Exercises

1. **Break the judge with a confident hallucination.** Write a candidate answer for the BGP question that is fluent, well-structured, cites "(RFC 4271)", and is factually wrong (e.g. claims AS_PATH is used for encryption). Judge it with both the anchored and unanchored judges. Does either give it a passing score?
2. **Measure position bias.** Build a pairwise variant of the judge: a tool schema with `winner: 'A' | 'B'`, given two candidates. Run it on the terse/padded pair twice, once with terse as A and once with terse as B, across all 13 answerable questions' generated answers vs. references. How often does the verdict flip with the ordering?
3. **Judge-model sweep.** Rerun the 16-case judging pass with `JUDGE_MODEL` set to `openai/gpt-oss-120b`, and compute score-by-score agreement with the 20B run. Where do they disagree: random cases, or systematically on the 3-vs-4 boundary?
4. **Add a `citations_valid` field.** Extend `JUDGE_TOOL` with a boolean that checks whether every RFC number cited in the candidate actually appears in the retrieved chunks (pass the chunk list into the judge prompt). This turns the judge into a faithfulness checker against sources, not just against the reference.

In [ ]:
# Step 1: read the first 10 judged outputs and form your own verdict on each.
sample = judged[:10]
for i, row in enumerate(sample):
    print('=' * 78)
    print(f"[{i}] {row['question']}")
    print(f"    answer : {row['answer'][:300]}")
    print(f"    judge  : score={row['grade']['score']} "
          f"faithful={row['grade']['faithful']}")


In [ ]:
# Step 2: record YOUR verdict for each index above.
#   True  = you would accept this answer (equivalent to score >= 4)
#   False = you would not
#   None  = skip
HUMAN_LABELS = {0: None, 1: None, 2: None, 3: None, 4: None,
                5: None, 6: None, 7: None, 8: None, 9: None}

labeled = {i: v for i, v in HUMAN_LABELS.items() if v is not None}
if not labeled:
    print('Fill in HUMAN_LABELS above, then rerun this cell.')
else:
    agree = sum((sample[i]['grade']['score'] >= 4) == v for i, v in labeled.items())
    print(f'agreement with judge: {agree}/{len(labeled)} = {agree / len(labeled):.0%}')
    for i, v in labeled.items():
        jv = sample[i]['grade']['score'] >= 4
        if jv != v:
            print(f'  DISAGREE [{i}] you={v} judge={jv} '
                  f'(score {sample[i]["grade"]["score"]}): '
                  f'{sample[i]["question"][:55]}')


Treat the disagreements as the interesting output, not the agreement rate. Each disagreement is either a judge bug (fix the rubric; usually the anchor definitions are too loose at the boundary you disagreed on) or a labeling insight (your own bar was fuzzy, so tighten the reference answer). Below ~80% agreement, don't ship decisions on this judge's numbers.

One habit that scales: keep the hand-labeled examples. They become a *meta-eval*, a golden set for the judge itself, that you rerun whenever the judge prompt or model changes. Hosted eval tools (Braintrust, LangSmith, promptfoo and friends) all have UIs for exactly this labeling loop; the mechanics are what you just did by hand.
